# Random Forest Classifier

Part 1: Predicting if a Storm Would Occur using Random Forest Classifier

Input: "Year", "MONTH", "DAY", "CO2 emission (Tons)", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov","Dec"
Objective: min Z1 classification error, when predicting tropical storms
Constraints: CO2 emissions, Monthly global temperatures, Year, Month, and Day >= 0 and If Storm Intensity Label > 1 && Storm Intensity Label >=7 then “storm_occured” = 1 else 0

Expected output - binary value 1 for tropical storm occurred else 0

In [55]:
#all imports
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.utils import shuffle
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
import time
from sklearn.metrics import accuracy_score, precision_score, mean_squared_error, mean_absolute_error, r2_score, classification_report
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline as ImbPipeline

In [56]:
#Reading from the dataset
data = pd.read_csv("completed_dataset_for_IS_project_25.csv")

#testing if the reading from the dataset was successful
print(data.head(5))

   Year  MONTH  DAY   LAT  LONG  WIND_KTS  PRESSURE CAT  Shape_Leng Country  \
0  1880      8   11  23.0 -91.9        70         0  H1    0.806226  Mexico   
1  1880      8   11  23.4 -92.6        80         0  H1    0.761577  Mexico   
2  1880      8   11  23.7 -93.3        80         0  H1    0.583095  Mexico   
3  1880      8   12  24.0 -93.8        90         0  H2    0.670820  Mexico   
4  1880      9    6  23.9 -88.6        40         0  TS    0.360555  Mexico   

   ...   Jun   Jul   Aug   Sep   Oct   Nov   Dec  Storm Intensity  \
0  ... -0.21 -0.18 -0.11 -0.15 -0.24 -0.22 -0.18         56.43582   
1  ... -0.21 -0.18 -0.11 -0.15 -0.24 -0.22 -0.18         60.92616   
2  ... -0.21 -0.18 -0.11 -0.15 -0.24 -0.22 -0.18         46.64760   
3  ... -0.21 -0.18 -0.11 -0.15 -0.24 -0.22 -0.18         60.37380   
4  ... -0.21 -0.18 -0.11 -0.15 -0.24 -0.22 -0.18         14.42220   

   Storm Intensity Label  Wind Speed Squared  
0                      3                4900  
1               

Using the data in the dataset where the Storm Intensity Label is greater than 1 and less than 7 to represent tropical storm records the the lables less than and equal to 1 to represent the data for no storm occured 

In [57]:
data["storm_occured"] = data["Storm Intensity Label"].apply(lambda val: 1 if 1 < val <= 7 else 0)

Checking if the no storm data was added into the dataset

In [58]:
no_storm = data[data["storm_occured"]== 0]
yes_storm = data[data["storm_occured"]== 1]

print(no_storm.shape, yes_storm.shape)

(14017, 27) (38639, 27)


The data is unbalanced leaning to the yes storm data which would cause our model to have a bias to yes storm, will have to balance the data for a better prediction 

In [59]:
yes_storm = yes_storm.head(no_storm.shape[0])
print(no_storm.shape, yes_storm.shape)

(14017, 27) (14017, 27)


Joining and shuffling the remaining data

In [60]:
data = pd.concat([no_storm, yes_storm])
data = shuffle(data)
data.head(5)

,Year,MONTH,DAY,LAT,LONG,WIND_KTS,PRESSURE,CAT,Shape_Leng,Country,...,Jul,Aug,Sep,Oct,Nov,Dec,Storm Intensity,Storm Intensity Label,Wind Speed Squared,storm_occured
3331,1893,9,9,34.8,-88.5,30,0,TD,0.300000,United States,...,-0.14,-0.25,-0.22,-0.19,-0.19,-0.34,9.00000,1,900,0
11891,1934,5,29,33.6,-81.5,35,0,TS,0.447214,United States,...,-0.10,-0.12,-0.15,-0.06,0.04,-0.02,15.65249,2,1225,1
6573,1908,10,15,11.7,-80.0,45,0,TS,0.300000,Colombia,...,-0.34,-0.46,-0.35,-0.43,-0.52,-0.49,13.50000,2,2025,1
34822,1986,7,25,18.2,-164.2,30,0,TD,1.019804,United States,...,0.11,0.16,0.03,0.15,0.10,0.13,30.59412,1,900,0
10398,1928,9,18,32.5,-80.8,60,978,TS,1.000000,United States,...,-0.19,-0.22,-0.21,-0.19,-0.09,-0.16,60.00000,2,3600,1


Spliting the data

In [61]:
features = ["Year", "MONTH", "DAY", "CO2 emission (Tons)", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov","Dec"]
target = "storm_occured"

X= data[features]
y = data[target]

#Spliting the merged dataset into training (70%), validation (15%), and testing (15%) sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=5, stratify=y) #for balance split
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=5, stratify=y_temp)

#testing splits
print(f"Total size: {len(X)}")
print(f"Train size: {len(X_train)}")
print(f"Validation size: {len(X_val)}")
print(f"Test size: {len(X_test)}")

Total size: 28034
Train size: 19623
Validation size: 4205
Test size: 4206


Training the Model

In [62]:
#dropping rows with missing values
X_train, y_train = X_train.dropna(), y_train.loc[X_train.dropna().index]
X_val, y_val = X_val.dropna(), y_val.loc[X_val.dropna().index]
X_test, y_test = X_test.dropna(), y_test.loc[X_test.dropna().index]

print(y_train.value_counts())

model = RandomForestClassifier(n_estimators=100, random_state=5, max_features= 5)
model.fit(X_train, y_train)

storm_occured
0    9812
1    9811
Name: count, dtype: int64


RandomForestClassifier(max_features=5, random_state=5)

Evaluation using the test and validation data

Classification → Accuracy, Precision
Efficiency → Latency


In [63]:
#Test
start_time_test = time.time()
y_test_prediction = model.predict(X_test)
latency_test = time.time() - start_time_test

accuracy_test = accuracy_score(y_test, y_test_prediction)
precision_test = precision_score(y_test, y_test_prediction)

print("Evaluation Test")
print(f"Accuracy: {accuracy_test}")
print(f"Precision: {precision_test}")
print(f"Latency: {latency_test}")

Evaluation Test
Accuracy: 0.9652876842605801
Precision: 0.9566028931404573
Latency: 0.07043147087097168


In [64]:
#Validation
start_time_val = time.time()
y_val_prediction = model.predict(X_val)
latency_val = time.time() - start_time_val

accuracy_val = accuracy_score(y_val, y_val_prediction)
precision_val = precision_score(y_val, y_val_prediction)

print("Evaluation Validation")
print(f"Accuracy: {accuracy_val}")
print(f"Precision: {precision_val}")
print(f"Latency: {latency_val}")

Evaluation Validation
Accuracy: 0.9657550535077289
Precision: 0.954524361948956
Latency: 0.06577706336975098


For the user to add in their variables to predict if a storm would occur or not and if a storm does occur it would lead the user into the other model to predict the intensity

In [65]:
'''
def predict_storm(model, year, month, day, co2, temps_dict):
    input = {
        "Year": [year],
        "MONTH": [month],
        "DAY": [day],
        "CO2 emission (Tons)": [co2],
    }

    months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov","Dec"]
    
    for month_temp in months:
        input[month_temp] = [temps_dict.get(month_temp, 0)]

    input_data = pd.DataFrame(input)
    prediction = model.predict(input_data)[0]

    return prediction #0 or 1 for the model in part 2
'''

'\ndef predict_storm(model, year, month, day, co2, temps_dict):\n    input = {\n        "Year": [year],\n        "MONTH": [month],\n        "DAY": [day],\n        "CO2 emission (Tons)": [co2],\n    }\n\n    months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov","Dec"]\n    \n    for month_temp in months:\n        input[month_temp] = [temps_dict.get(month_temp, 0)]\n\n    input_data = pd.DataFrame(input)\n    prediction = model.predict(input_data)[0]\n\n    return prediction #0 or 1 for the model in part 2\n'

In [66]:
'''
#example for predicting a possible storm
possible_temps= {"Jan": -0.19, "Feb": -0.25, "Mar": -0.09, "Apr": -0.17, "May": -0.10, "Jun": -0.21, "Jul": -0.18, "Aug": -0.11, "Sep": -0.15, "Oct": -0.24, "Nov": -0.22, "Dec": -0.18}
will_storm_occur = predict_storm(model=model, year=1880, month=8, day=9, co2= 0.0, temps_dict = possible_temps)
print("A storm was predicted" if will_storm_occur == 1 else "No storm was predicted")
'''

'\n#example for predicting a possible storm\npossible_temps= {"Jan": -0.19, "Feb": -0.25, "Mar": -0.09, "Apr": -0.17, "May": -0.10, "Jun": -0.21, "Jul": -0.18, "Aug": -0.11, "Sep": -0.15, "Oct": -0.24, "Nov": -0.22, "Dec": -0.18}\nwill_storm_occur = predict_storm(model=model, year=1880, month=8, day=9, co2= 0.0, temps_dict = possible_temps)\nprint("A storm was predicted" if will_storm_occur == 1 else "No storm was predicted")\n'

In [67]:
'''
#example for predicting a no storm
possible_temps= {"Jan": -1.5, "Feb": -1.3, "Mar": -1.1, "Apr": -1.2, "May": -1.0, "Jun": -1.4, "Jul": -1.3, "Aug": -1.5, "Sep": -1.2, "Oct": -1.3, "Nov": -1.4, "Dec": -1.6}
will_storm_occur = predict_storm(model=model, year=1890, month=1, day=5, co2= 15000000.0, temps_dict = possible_temps)
print("A storm was predicted" if will_storm_occur == 1 else "No storm was predicted")
'''

'\n#example for predicting a no storm\npossible_temps= {"Jan": -1.5, "Feb": -1.3, "Mar": -1.1, "Apr": -1.2, "May": -1.0, "Jun": -1.4, "Jul": -1.3, "Aug": -1.5, "Sep": -1.2, "Oct": -1.3, "Nov": -1.4, "Dec": -1.6}\nwill_storm_occur = predict_storm(model=model, year=1890, month=1, day=5, co2= 15000000.0, temps_dict = possible_temps)\nprint("A storm was predicted" if will_storm_occur == 1 else "No storm was predicted")\n'

In [68]:
print("Classification Report for Model 1 - Gradient Boosting Classifier")

print("Test Report")
print(classification_report(y_test, y_test_prediction))

print("Validation Report")
print(classification_report(y_val, y_val_prediction))

Classification Report for Model 1 - Gradient Boosting Classifier
Test Report
              precision    recall  f1-score   support

           0       0.97      0.96      0.96      2103
           1       0.96      0.97      0.97      2103

    accuracy                           0.97      4206
   macro avg       0.97      0.97      0.97      4206
weighted avg       0.97      0.97      0.97      4206

Validation Report
              precision    recall  f1-score   support

           0       0.98      0.95      0.97      2102
           1       0.95      0.98      0.97      2103

    accuracy                           0.97      4205
   macro avg       0.97      0.97      0.97      4205
weighted avg       0.97      0.97      0.97      4205



# Random Forest Regressor

Part 2: Predicting the intensity of the storm using Random Forest Regressor

Input: "Wind Speed Squared", "PRESSURE", "CO2 emission (Tons)", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov","Dec"
Objective: min Z2 error in predicting the intensity of the tropical storm
Constraints: Wind speed, Pressure >= 0 and stormed_occured = 1

In [69]:
#Only keeping the rows we need
data = data[["storm_occured", "Storm Intensity Label", "Wind Speed Squared", "PRESSURE", "CAT", "CO2 emission (Tons)", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov","Dec"]]

In [70]:
#Only using the rows were storm_occured is 1
storm_data = data[data["storm_occured"]==1]

Ensures the intensity model trains only where tropical storms occur by filtering to 'Storm Intensity Label' > 1

In [71]:
storm_data = storm_data[(storm_data["Storm Intensity Label"] > 1) & (storm_data["Storm Intensity Label"] <= 7)]

In [72]:
#Dropping rows that are missing values from our inputs
storm_data = storm_data.dropna(subset= ["Wind Speed Squared", "PRESSURE", "CAT", "Storm Intensity Label", "CO2 emission (Tons)", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov","Dec"])

Spliting the data

In [73]:
#Reading the updated csv 
data = pd.read_csv("completed_dataset_for_IS_project_25.csv")

features = ["Wind Speed Squared", "PRESSURE", "CO2 emission (Tons)", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov","Dec"]
target = "Storm Intensity Label"

X= data[features]
y = data[target]

#Spliting the dataset into training (70%), validation (15%), and testing (15%) sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=5)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=5)

#testing splits
print(f"Total size: {len(X)}")
print(f"Train size: {len(X_train)}")
print(f"Validation size: {len(X_val)}")
print(f"Test size: {len(X_test)}")

Total size: 52656
Train size: 36859
Validation size: 7898
Test size: 7899


In [74]:
#Pipeline

intensity_pipeline = ImbPipeline([
        ('scaler', StandardScaler()),
        ('classifier', RandomForestRegressor(random_state=42))
    ])

#Hyperparameter tuning to ajust tree depth and ensemble sizes for Random Forrest classifier model
param_grid_int = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [None, 10]
}

grid_search_int = GridSearchCV(
    intensity_pipeline,
    param_grid_int,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=2,
    verbose=2,
    error_score='raise'
)

grid_search_int.fit(X_train, y_train)
best_intensity_model = grid_search_int.best_estimator_

Fitting 5 folds for each of 4 candidates, totalling 20 fits


Evaluation for the test and validation

Regression → MSE, MAE, R-squared 
Efficiency → Latency

In [75]:
#Test

test_pred_intensity = best_intensity_model.predict(X_test)
test_mse = mean_squared_error(y_test, test_pred_intensity)
test_mae = mean_absolute_error(y_test, test_pred_intensity)
test_r2 = r2_score(y_test, test_pred_intensity)

print("Intensity Test Evaluation:")
print(f"Mean Squared Error (MSE): {test_mse:.2f}")
print(f"Mean Absolute Error (MAE): {test_mae:.2f}")
print(f"R-squared (R2): {test_r2:.2f}")

Intensity Test Evaluation:
Mean Squared Error (MSE): 0.27
Mean Absolute Error (MAE): 0.16
R-squared (R2): 0.88


In [76]:
#Validation

val_pred_intensity = best_intensity_model.predict(X_val)
val_mse = mean_squared_error(y_val, val_pred_intensity)
val_mae = mean_absolute_error(y_val, val_pred_intensity)
val_r2 = r2_score(y_val, val_pred_intensity)

print("Intensity Validation Evaluation:")
print(f"Mean Squared Error (MSE): {val_mse:.2f}")
print(f"Mean Absolute Error (MAE): {val_mae:.2f}")
print(f"R-squared (R2): {val_r2:.2f}")

Intensity Validation Evaluation:
Mean Squared Error (MSE): 0.26
Mean Absolute Error (MAE): 0.16
R-squared (R2): 0.87


In [77]:
def predict_storm(model, intensity_model, year, month, day, co2, temps_dict, wind_speed_squared, pressure):
    input = {
        "Year": [year],
        "MONTH": [month],
        "DAY": [day],
        "CO2 emission (Tons)": [co2],
    }

    months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov","Dec"]
    
    for month_temp in months:
        input[month_temp] = [temps_dict.get(month_temp, 0)]

    input_data = pd.DataFrame(input)
    prediction = model.predict(input_data)[0]

    if prediction == 1:
        avg_temp = sum(temps_dict.values())/len(temps_dict)

        intensity ={
            "Wind Speed Squared": [wind_speed_squared],
            "PRESSURE": [pressure],
            "CO2 emission (Tons)": [co2],
        }

        for month_temp in months:
            intensity[month_temp] = [temps_dict.get(month_temp, avg_temp)]

        intensity_data = pd.DataFrame(intensity)
        intensity_prediction =round(intensity_model.predict(intensity_data)[0])

        if 2<= intensity_prediction <= 7:
            return 1
        else:
            return 0 #low intensity_prediction
    else: 
        return 0 #no storm

    #return prediction #0 or 1 for the model in part 2

In [78]:
def predict_intensity(model, wind_speed_squared, pressure, co2, temps_dict):
    
    months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov","Dec"]

    input = {
        "Wind Speed Squared": wind_speed_squared,
        "PRESSURE": pressure, 
        "CO2 emission (Tons)": co2,
    }

    avg_temp = sum(temps_dict.values())/len(temps_dict)

    for month_temp in months:
        input[month_temp] = [temps_dict.get(month_temp, avg_temp)]

    input_data = pd.DataFrame(input)
    prediction = model.predict(input_data)[0]

    return prediction

In [79]:
#example, a storm is predicted
possible_temps= {"Jan": -0.19, "Feb": -0.25, "Mar": -0.09, "Apr": -0.17, "May": -0.10, "Jun": -0.21, "Jul": -0.18, "Aug": -0.11, "Sep": -0.15, "Oct": -0.24, "Nov": -0.22, "Dec": -0.18}
will_storm_occur = predict_storm(model=model, intensity_model= best_intensity_model, year=1880, month=8, day=9, co2= 0.0, temps_dict = possible_temps, wind_speed_squared=1600.0, pressure= 960.0)
#print("A storm was predicted" if will_storm_occur == 1 else "No storm was predicted")

if will_storm_occur == 1:
    print("A storm was predicted, now predicting it's intensity...")
    #intensity = predict_intensity(model = model_2, wind_kts=70, cat= "H2", co2=412.5, year=2024, month=8, day=14)
    intensity = predict_intensity(model = best_intensity_model, wind_speed_squared=1600.0, pressure= 960.0, co2=0.0, temps_dict = possible_temps)
    #intensity = predict_intensity(model = best_intensity_model, co2=412.5, year=2024, month=8, day=14, temps_dict = possible_temps)

    #Before rounding
    #print(f"Predicted storm intensity: {intensity}")
    #print("Based on MyNASEData Hurricane Dynamics")
    
    #Some times I get the intensity as a decimal, using this to round to the nearest whole number
    intensity = round(intensity)
    
    if intensity == -2:
        value = "Warning flag"
    elif intensity == -1:
        value = "Extra-tropical (non-standard storm)"
    elif intensity == 0:
        value = "Low-pressure system"
    elif intensity == 1:
        value = "Tropical Depression"
    elif intensity == 2:
        value = "Tropical Storm"
    elif intensity == 3:
        value = "Hurricane Category 1"
    elif intensity == 4:
        value = "Hurricane Category 2"
    elif intensity == 5:
        value = "Hurricane Category 3"
    elif intensity == 6:
        value = "Hurricane Category 4"
    elif intensity == 7:
        value = "Hurricane Category 5"
    else:
        value = "Unknown"

    #print(f"Predicted storm intensity: {value} with an intensity of {intensity}")
    print(f"Predicted storm intensity: {value}")
else:
    print(f"No storm was predicted, skipping the intensity prediction")

A storm was predicted, now predicting it's intensity...
Predicted storm intensity: Tropical Storm


In [80]:
#example, a no storm is predicted
possible_temps= {"Jan": -0.05, "Feb": -0.06, "Mar": -0.04, "Apr": -0.03, "May": -0.02, "Jun": -0.01, "Jul": -0.2, "Aug": -0.03, "Sep": -0.04, "Oct": -0.05, "Nov": -0.06, "Dec": -0.04}
will_storm_occur = predict_storm(model=model, intensity_model= best_intensity_model, year=1890, month=9, day=15, co2= 100000.0, temps_dict = possible_temps, wind_speed_squared=100.0, pressure= 1015.0)
#print("A storm was predicted" if will_storm_occur == 1 else "No storm was predicted")

if will_storm_occur == 1:
    print("A storm was predicted, now predicting it's intensity...")
    #intensity = predict_intensity(model = model_2, wind_kts=70, cat= "H2", co2=412.5, year=2024, month=8, day=14)
    intensity = predict_intensity(model = best_intensity_model, wind_speed_squared=100.0, pressure= 1015.0, co2=100000.0, temps_dict = possible_temps)
    #intensity = predict_intensity(model = best_intensity_model, co2=412.5, year=2024, month=8, day=14, temps_dict = possible_temps)

    #Before rounding
    #print(f"Predicted storm intensity: {intensity}")
    #print("Based on MyNASEData Hurricane Dynamics")
    
    #Some times I get the intensity as a decimal, using this to round to the nearest whole number
    intensity = round(intensity)
    
    if intensity == -2:
        value = "Warning flag"
    elif intensity == -1:
        value = "Extra-tropical (non-standard storm)"
    elif intensity == 0:
        value = "Low-pressure system"
    elif intensity == 1:
        value = "Tropical Depression"
    elif intensity == 2:
        value = "Tropical Storm"
    elif intensity == 3:
        value = "Hurricane Category 1"
    elif intensity == 4:
        value = "Hurricane Category 2"
    elif intensity == 5:
        value = "Hurricane Category 3"
    elif intensity == 6:
        value = "Hurricane Category 4"
    elif intensity == 7:
        value = "Hurricane Category 5"
    else:
        value = "Unknown"

    #print(f"Predicted storm intensity: {value} with an intensity of {intensity}")
    print(f"Predicted storm intensity: {value}")
else:
    print(f"No storm was predicted, skipping the intensity prediction")

No storm was predicted, skipping the intensity prediction
